In [2]:
import sys
sys.path.append('../../')
import json
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from utilities import load_embedding
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from utilities import print_exams

In [3]:
class fluProfiler_Dataset(Dataset):
    def __init__(self, DataFrame):
        self.emb_file_name_a = ('matrix_' + DataFrame['seq_id_a']).tolist()
        self.emb_file_name_b = ('matrix_' + DataFrame['seq_id_b']).tolist()
        self.emb_file_name_c = ('matrix_' + DataFrame['seq_id_c']).tolist()
        self.emb_file_name_d = ('matrix_' + DataFrame['seq_id_d']).tolist()

        self.strainPassCats = convert_Pass2tensor(('<cls>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>').tolist())

        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.emb_file_name_a[idx], self.emb_file_name_b[idx], self.emb_file_name_c[idx], self.emb_file_name_d[idx], \
               self.strainPassCats[idx], self.labels[idx]

def convert_Pass2tensor(pass_cats):
    result = [
        item.replace('<cls>', '0').replace('<eos>', '1').replace('<EGG>', '2').replace('<CELL>', '3').replace('<BOTH>', '4')
        for item in pass_cats
    ]
    result = torch.tensor([[int(number) for number in [char for char in item]] for item in result])
    return result

def generate_matrix(matrix_list):
    seq_len = [mat.shape[0] for mat in matrix_list]
    max_len = max(seq_len)
    mask_list = []
    for i in range(len(matrix_list)): 
        matrix_list[i] = F.pad(matrix_list[i], (0, 0, 0, max_len - seq_len[i]))
        mask = torch.concat((torch.ones(1,seq_len[i]),torch.zeros(1,max_len-seq_len[i])),axis=1)
        mask_list.append(mask)
    matrix = torch.stack(matrix_list)
    mask = torch.stack(mask_list).view(len(matrix_list),max_len)
    return matrix, mask


In [4]:
device = torch.device('cuda:0')
test_data = pd.read_csv('../../../data/data_40/titer/test.csv')
test_dataset = fluProfiler_Dataset(test_data)
test_dataloader = DataLoader(test_dataset, batch_size=100, shuffle=False)

In [5]:
Crick_41 = pd.read_csv('../../../data/data_40/Crick_41_mapped.csv')
group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
agg_dict = {c: 'first' for c in Crick_41.columns if c not in group_columns}
agg_dict['label'] = 'mean'
test_data_41 = Crick_41.groupby(group_columns).agg(agg_dict).reset_index()
test_dataset_41 = fluProfiler_Dataset(test_data_41)
test_dataloader_41 = DataLoader(test_dataset_41, batch_size=128, shuffle=False, num_workers=4)

## 1. LucaVirus embedding

In [ ]:
model = torch.load('../../../trained_model/1.7_Artificial_back/2025-08-19_17-43-32.pth', weights_only=False)

### Internal test

In [7]:
# load embedding
embedding_df = test_data
sequence_names = pd.concat([embedding_df['seq_id_a'], embedding_df['seq_id_b'], 
                            embedding_df['seq_id_c'], embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
emb_dict_lucavirus = load_embedding("../../../data/data_40/embedding_Crick", files=sequence_names)

Loading tensor: 100%|██████████| 6323/6323 [18:10<00:00,  5.80file/s]


In [8]:
prediction_ls_test = []
reference_ls_test = []
logits_ls = []
loss_ls_test = []
model.eval()
for batch in test_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch

    matrixs_a, masks_a = generate_matrix([emb_dict_lucavirus[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict_lucavirus[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict_lucavirus[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict_lucavirus[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c,
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b,
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats,
                                        labels=labels)

    loss_ls_test.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls_test.extend(output.view(-1).tolist())
    reference_ls_test.extend(labels.tolist())

test_data['prediction_lucavirus'] = prediction_ls_test

In [9]:
All_result = print_exams(test_data['prediction_lucavirus'], test_data['label'])
H1_result = print_exams(test_data.loc[test_data['Type'] == 'H1N1', 'prediction_lucavirus'], test_data.loc[test_data['Type'] == 'H1N1', 'label'])
H3_result = print_exams(test_data.loc[test_data['Type'] == 'H3N2', 'prediction_lucavirus'], test_data.loc[test_data['Type'] == 'H3N2', 'label'])

MAE: 0.59523
MSE: 0.60702
pearson correlation: 0.92036
spearman correlation: 0.89742
R2_score: 0.84483
MAE: 0.57682
MSE: 0.56688
pearson correlation: 0.90677
spearman correlation: 0.80060
R2_score: 0.82013
MAE: 0.61558
MSE: 0.65138
pearson correlation: 0.90176
spearman correlation: 0.90210
R2_score: 0.80917


### 41 test

In [10]:
# load embedding
embedding_df = test_data_41
sequence_names = pd.concat([embedding_df['seq_id_a'], embedding_df['seq_id_b'], 
                            embedding_df['seq_id_c'], embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
emb_dict_lucavirus_41 = load_embedding("../../../data/data_40/embedding_41", files=sequence_names)

Loading tensor: 100%|██████████| 469/469 [01:08<00:00,  6.85file/s]


In [11]:
prediction_ls_test = []
reference_ls_test = []
logits_ls = []
loss_ls_test = []
model.eval()
for batch in test_dataloader_41:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch

    matrixs_a, masks_a = generate_matrix([emb_dict_lucavirus_41[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict_lucavirus_41[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict_lucavirus_41[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict_lucavirus_41[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c,
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b,
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats,
                                        labels=labels)

    loss_ls_test.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls_test.extend(output.view(-1).tolist())
    reference_ls_test.extend(labels.tolist())

test_data_41['prediction_lucavirus'] = prediction_ls_test

In [13]:
All_result_41 = print_exams(test_data_41['prediction_lucavirus'], test_data_41['label'])
H1_result_41 = print_exams(test_data_41.loc[test_data_41['Type'] == 'H1N1', 'prediction_lucavirus'], test_data_41.loc[test_data_41['Type'] == 'H1N1', 'label'])
H3_result_41 = print_exams(test_data_41.loc[test_data_41['Type'] == 'H3N2', 'prediction_lucavirus'], test_data_41.loc[test_data_41['Type'] == 'H3N2', 'label'])

MAE: 0.67645
MSE: 0.74275
pearson correlation: 0.70152
spearman correlation: 0.69498
R2_score: 0.47283
MAE: 0.62707
MSE: 0.62957
pearson correlation: 0.52130
spearman correlation: 0.52141
R2_score: 0.16644
MAE: 0.73952
MSE: 0.88731
pearson correlation: 0.41373
spearman correlation: 0.38477
R2_score: 0.11089


## 2. LucaOne embedding

In [ ]:
model = torch.load('../../../trained_model/lucaOne_model/2025-11-17_05-33-35.pth', weights_only=False)

### Internal test

In [ ]:
# load embedding
embedding_df = test_data
sequence_names = pd.concat([embedding_df['seq_id_a'], embedding_df['seq_id_b'], 
                            embedding_df['seq_id_c'], embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
emb_dict_lucaone = load_embedding("../../../data/data_40/embedding_Crick_lucaOne", files=sequence_names)

Loading tensor: 100%|██████████| 6323/6323 [10:32<00:00, 10.00file/s]


In [17]:
prediction_ls_test = []
reference_ls_test = []
logits_ls = []
loss_ls_test = []
model.eval()
for batch in test_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch

    matrixs_a, masks_a = generate_matrix([emb_dict_lucaone[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict_lucaone[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict_lucaone[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict_lucaone[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c,
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b,
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats,
                                        labels=labels)

    loss_ls_test.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls_test.extend(output.view(-1).tolist())
    reference_ls_test.extend(labels.tolist())

test_data['prediction_lucaone'] = prediction_ls_test

In [20]:
All_result = print_exams(test_data['prediction_lucaone'], test_data['label'])
H1_result = print_exams(test_data.loc[test_data['Type'] == 'H1N1', 'prediction_lucaone'], test_data.loc[test_data['Type'] == 'H1N1', 'label'])
H3_result = print_exams(test_data.loc[test_data['Type'] == 'H3N2', 'prediction_lucaone'], test_data.loc[test_data['Type'] == 'H3N2', 'label'])

MAE: 0.60886
MSE: 0.62987
pearson correlation: 0.91607
spearman correlation: 0.89512
R2_score: 0.83899
MAE: 0.58090
MSE: 0.57759
pearson correlation: 0.90425
spearman correlation: 0.80631
R2_score: 0.81674
MAE: 0.63974
MSE: 0.68764
pearson correlation: 0.89462
spearman correlation: 0.89459
R2_score: 0.79855


### 41 test

In [24]:
# load embedding
embedding_df = test_data_41
sequence_names = pd.concat([embedding_df['seq_id_a'], embedding_df['seq_id_b'], 
                            embedding_df['seq_id_c'], embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
emb_dict_esm_41 = load_embedding("../../../data/data_40/embedding_41_lucaone", files=sequence_names)

Loading tensor: 100%|██████████| 469/469 [01:07<00:00,  6.98file/s]


In [27]:
prediction_ls_test = []
reference_ls_test = []
logits_ls = []
loss_ls_test = []
model.eval()
for batch in test_dataloader_41:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch

    matrixs_a, masks_a = generate_matrix([emb_dict_esm_41[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict_esm_41[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict_esm_41[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict_esm_41[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c,
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b,
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats,
                                        labels=labels)

    loss_ls_test.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls_test.extend(output.view(-1).tolist())
    reference_ls_test.extend(labels.tolist())

test_data_41['prediction_lucaone'] = prediction_ls_test

In [35]:
test_data_41[['label', 'prediction_lucavirus', 'prediction_lucaone']]

,label,prediction_lucavirus,prediction_lucaone
0,2.0,0.042759,-0.001865
1,-1.0,0.809032,1.135172
2,1.0,0.523131,0.391154
3,2.0,0.356234,0.402566
4,0.0,0.454289,0.813964
...,...,...,...
2081,3.0,1.963822,2.132412
2082,3.0,1.906541,2.093301
2083,2.0,1.794588,2.145077
2084,2.0,2.148864,1.940154


In [28]:
All_result_41 = print_exams(test_data_41['prediction_lucaone'], test_data_41['label'])
H1_result_41 = print_exams(test_data_41.loc[test_data_41['Type'] == 'H1N1', 'prediction_lucaone'], test_data_41.loc[test_data_41['Type'] == 'H1N1', 'label'])
H3_result_41 = print_exams(test_data_41.loc[test_data_41['Type'] == 'H3N2', 'prediction_lucaone'], test_data_41.loc[test_data_41['Type'] == 'H3N2', 'label'])

MAE: 0.73789
MSE: 0.85574
pearson correlation: 0.69811
spearman correlation: 0.68037
R2_score: 0.39263
MAE: 0.63031
MSE: 0.61764
pearson correlation: 0.50441
spearman correlation: 0.50951
R2_score: 0.18223
MAE: 0.87529
MSE: 1.15987
pearson correlation: 0.38377
spearman correlation: 0.36509
R2_score: -0.16222


## 3. ESM embedding

In [29]:
model = torch.load('../../../trained_model/esm3b_model/2025-11-18_02-24-53.pth', weights_only=False)

### Internal test

In [36]:
# load embedding
embedding_df = test_data
sequence_names = pd.concat([embedding_df['seq_id_a'], embedding_df['seq_id_b'], 
                            embedding_df['seq_id_c'], embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
emb_dict_esm = load_embedding("../../../data/data_40/embedding_Crick_esm3b", files=sequence_names)

Loading tensor: 100%|██████████| 6323/6323 [19:58<00:00,  5.28file/s]


In [37]:
prediction_ls_test = []
reference_ls_test = []
logits_ls = []
loss_ls_test = []
model.eval()
for batch in test_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch

    matrixs_a, masks_a = generate_matrix([emb_dict_esm[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict_esm[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict_esm[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict_esm[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c,
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b,
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats,
                                        labels=labels)

    loss_ls_test.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls_test.extend(output.view(-1).tolist())
    reference_ls_test.extend(labels.tolist())

test_data['prediction_esm3b'] = prediction_ls_test

In [38]:
All_result = print_exams(test_data['prediction_esm3b'], test_data['label'])
H1_result = print_exams(test_data.loc[test_data['Type'] == 'H1N1', 'prediction_esm3b'], test_data.loc[test_data['Type'] == 'H1N1', 'label'])
H3_result = print_exams(test_data.loc[test_data['Type'] == 'H3N2', 'prediction_esm3b'], test_data.loc[test_data['Type'] == 'H3N2', 'label'])

MAE: 0.60108
MSE: 0.62957
pearson correlation: 0.91679
spearman correlation: 0.89598
R2_score: 0.83906
MAE: 0.57677
MSE: 0.57623
pearson correlation: 0.90410
spearman correlation: 0.80792
R2_score: 0.81717
MAE: 0.62795
MSE: 0.68852
pearson correlation: 0.89566
spearman correlation: 0.89584
R2_score: 0.79829


### 41 test

In [39]:
# load embedding
embedding_df = test_data_41
sequence_names = pd.concat([embedding_df['seq_id_a'], embedding_df['seq_id_b'], 
                            embedding_df['seq_id_c'], embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
emb_dict_esm_41 = load_embedding("../../../data/data_40/embedding_41_esm3b", files=sequence_names)

Loading tensor: 100%|██████████| 469/469 [01:12<00:00,  6.49file/s]


In [40]:
prediction_ls_test = []
reference_ls_test = []
logits_ls = []
loss_ls_test = []
model.eval()
for batch in test_dataloader_41:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch

    matrixs_a, masks_a = generate_matrix([emb_dict_esm_41[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict_esm_41[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict_esm_41[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict_esm_41[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c,
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b,
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats,
                                        labels=labels)

    loss_ls_test.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls_test.extend(output.view(-1).tolist())
    reference_ls_test.extend(labels.tolist())

test_data_41['prediction_esm3b'] = prediction_ls_test

In [41]:
All_result_41 = print_exams(test_data_41['prediction_esm3b'], test_data_41['label'])
H1_result_41 = print_exams(test_data_41.loc[test_data_41['Type'] == 'H1N1', 'prediction_esm3b'], test_data_41.loc[test_data_41['Type'] == 'H1N1', 'label'])
H3_result_41 = print_exams(test_data_41.loc[test_data_41['Type'] == 'H3N2', 'prediction_esm3b'], test_data_41.loc[test_data_41['Type'] == 'H3N2', 'label'])

MAE: 0.78925
MSE: 0.98886
pearson correlation: 0.61736
spearman correlation: 0.60821
R2_score: 0.29814
MAE: 0.76176
MSE: 0.92673
pearson correlation: 0.38961
spearman correlation: 0.40877
R2_score: -0.22700
MAE: 0.82437
MSE: 1.06823
pearson correlation: 0.33513
spearman correlation: 0.31426
R2_score: -0.07039


## 4. test

### internal

In [ ]:
# test_data.to_csv('Crick_early.csv')
# test_data_41.to_csv('Crick_41.csv')

In [8]:
test_data = pd.read_csv('Crick_early.csv')
test_data_41 = pd.read_csv('Crick_41.csv')

In [9]:
All_result = print_exams(test_data['prediction_lucavirus'], test_data['label'])
H1_result = print_exams(test_data.loc[test_data['Type'] == 'H1N1', 'prediction_lucavirus'], test_data.loc[test_data['Type'] == 'H1N1', 'label'])
H3_result = print_exams(test_data.loc[test_data['Type'] == 'H3N2', 'prediction_lucavirus'], test_data.loc[test_data['Type'] == 'H3N2', 'label'])

MAE: 0.59523
MSE: 0.60702
pearson correlation: 0.92036
spearman correlation: 0.89742
R2_score: 0.84483
MAE: 0.57682
MSE: 0.56688
pearson correlation: 0.90677
spearman correlation: 0.80060
R2_score: 0.82013
MAE: 0.61558
MSE: 0.65138
pearson correlation: 0.90176
spearman correlation: 0.90210
R2_score: 0.80917


In [10]:
All_result = print_exams(test_data['prediction_lucaone'], test_data['label'])
H1_result = print_exams(test_data.loc[test_data['Type'] == 'H1N1', 'prediction_lucaone'], test_data.loc[test_data['Type'] == 'H1N1', 'label'])
H3_result = print_exams(test_data.loc[test_data['Type'] == 'H3N2', 'prediction_lucaone'], test_data.loc[test_data['Type'] == 'H3N2', 'label'])

MAE: 0.60886
MSE: 0.62987
pearson correlation: 0.91607
spearman correlation: 0.89512
R2_score: 0.83899
MAE: 0.58090
MSE: 0.57759
pearson correlation: 0.90425
spearman correlation: 0.80631
R2_score: 0.81674
MAE: 0.63974
MSE: 0.68764
pearson correlation: 0.89462
spearman correlation: 0.89459
R2_score: 0.79855


In [11]:
All_result = print_exams(test_data['prediction_esm3b'], test_data['label'])
H1_result = print_exams(test_data.loc[test_data['Type'] == 'H1N1', 'prediction_esm3b'], test_data.loc[test_data['Type'] == 'H1N1', 'label'])
H3_result = print_exams(test_data.loc[test_data['Type'] == 'H3N2', 'prediction_esm3b'], test_data.loc[test_data['Type'] == 'H3N2', 'label'])

MAE: 0.60108
MSE: 0.62957
pearson correlation: 0.91679
spearman correlation: 0.89598
R2_score: 0.83906
MAE: 0.57677
MSE: 0.57623
pearson correlation: 0.90410
spearman correlation: 0.80792
R2_score: 0.81717
MAE: 0.62795
MSE: 0.68852
pearson correlation: 0.89566
spearman correlation: 0.89584
R2_score: 0.79829


### external

In [12]:
All_result = print_exams(test_data_41['prediction_lucavirus'], test_data_41['label'])
H1_result = print_exams(test_data_41.loc[test_data_41['Type'] == 'H1N1', 'prediction_lucavirus'], test_data_41.loc[test_data_41['Type'] == 'H1N1', 'label'])
H3_result = print_exams(test_data_41.loc[test_data_41['Type'] == 'H3N2', 'prediction_lucavirus'], test_data_41.loc[test_data_41['Type'] == 'H3N2', 'label'])

MAE: 0.67645
MSE: 0.74275
pearson correlation: 0.70152
spearman correlation: 0.69513
R2_score: 0.47283
MAE: 0.62707
MSE: 0.62957
pearson correlation: 0.52130
spearman correlation: 0.52141
R2_score: 0.16644
MAE: 0.73952
MSE: 0.88731
pearson correlation: 0.41373
spearman correlation: 0.38537
R2_score: 0.11089


In [13]:
All_result = print_exams(test_data_41['prediction_lucaone'], test_data_41['label'])
H1_result = print_exams(test_data_41.loc[test_data_41['Type'] == 'H1N1', 'prediction_lucaone'], test_data_41.loc[test_data_41['Type'] == 'H1N1', 'label'])
H3_result = print_exams(test_data_41.loc[test_data_41['Type'] == 'H3N2', 'prediction_lucaone'], test_data_41.loc[test_data_41['Type'] == 'H3N2', 'label'])

MAE: 0.73789
MSE: 0.85574
pearson correlation: 0.69811
spearman correlation: 0.68054
R2_score: 0.39263
MAE: 0.63031
MSE: 0.61764
pearson correlation: 0.50441
spearman correlation: 0.50951
R2_score: 0.18223
MAE: 0.87529
MSE: 1.15987
pearson correlation: 0.38377
spearman correlation: 0.36581
R2_score: -0.16222


In [14]:
All_result = print_exams(test_data_41['prediction_esm3b'], test_data_41['label'])
H1_result = print_exams(test_data_41.loc[test_data_41['Type'] == 'H1N1', 'prediction_esm3b'], test_data_41.loc[test_data_41['Type'] == 'H1N1', 'label'])
H3_result = print_exams(test_data_41.loc[test_data_41['Type'] == 'H3N2', 'prediction_esm3b'], test_data_41.loc[test_data_41['Type'] == 'H3N2', 'label'])

MAE: 0.78925
MSE: 0.98886
pearson correlation: 0.61736
spearman correlation: 0.60830
R2_score: 0.29814
MAE: 0.76176
MSE: 0.92673
pearson correlation: 0.38961
spearman correlation: 0.40877
R2_score: -0.22700
MAE: 0.82437
MSE: 1.06823
pearson correlation: 0.33513
spearman correlation: 0.31423
R2_score: -0.07039
